In [0]:
# ============================================
# Stage 4.3: Load DimSalesPerson with PySpark
# ============================================

from pyspark.sql import functions as F

# 读取 SalesPerson 和 Person
sp_path = "abfss://raw@stadventureworksdw.dfs.core.windows.net/AdventureWorks2019/SalesPerson/"
person_path = "abfss://raw@stadventureworksdw.dfs.core.windows.net/AdventureWorks2019/Person/"

df_sp = spark.read.parquet(sp_path)
df_person = spark.read.parquet(person_path)

print(f"SalesPerson rows: {df_sp.count()}")
print(f"Person rows: {df_person.count()}")

print("\n--- SalesPerson Schema ---")
df_sp.printSchema()

print("\n--- Person Schema ---")
df_person.printSchema()

SalesPerson rows: 17
Person rows: 19972

--- SalesPerson Schema ---
root
 |-- BusinessEntityID: integer (nullable = true)
 |-- TerritoryID: integer (nullable = true)
 |-- SalesQuota: decimal(19,4) (nullable = true)
 |-- Bonus: decimal(19,4) (nullable = true)
 |-- CommissionPct: decimal(10,4) (nullable = true)
 |-- SalesYTD: decimal(19,4) (nullable = true)
 |-- SalesLastYear: decimal(19,4) (nullable = true)
 |-- rowguid: string (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)


--- Person Schema ---
root
 |-- BusinessEntityID: integer (nullable = true)
 |-- PersonType: string (nullable = true)
 |-- NameStyle: boolean (nullable = true)
 |-- Title: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- MiddleName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Suffix: string (nullable = true)
 |-- EmailPromotion: integer (nullable = true)
 |-- AdditionalContactInfo: string (nullable = true)
 |-- Demographics: string (nullable = true)


In [0]:
# 转换：JOIN SalesPerson + Person
dim_salesperson = (df_sp
    .join(df_person, on="BusinessEntityID", how="inner")
    .select(
        F.col("BusinessEntityID").cast("int").alias("SalesPersonID"),
        F.concat_ws(" ", F.col("FirstName"), F.col("LastName")).alias("SalesPersonName"),
        F.col("TerritoryID").cast("int").alias("TerritoryID"),
        F.current_timestamp().alias("valid_from"),
        F.to_date(F.lit("9999-12-31")).alias("valid_to"),
        F.lit(1).cast("int").alias("is_current")
    )
    .withColumn("SalesPersonKey", F.monotonically_increasing_id() + 1)
    .select("SalesPersonKey", "SalesPersonID", "SalesPersonName", 
            "TerritoryID", "valid_from", "valid_to", "is_current")
)

display(dim_salesperson)
print(f"Total rows: {dim_salesperson.count()}")

SalesPersonKey,SalesPersonID,SalesPersonName,TerritoryID,valid_from,valid_to,is_current
1,274,Stephen Jiang,null,2026-09-27T07:02:30.633Z,9999-12-31,1
2,275,Michael Blythe,2,2026-09-27T07:02:30.633Z,9999-12-31,1
3,276,Linda Mitchell,4,2026-09-27T07:02:30.633Z,9999-12-31,1
4,277,Jillian Carson,3,2026-09-27T07:02:30.633Z,9999-12-31,1
5,278,Garrett Vargas,6,2026-09-27T07:02:30.633Z,9999-12-31,1
6,279,Tsvi Reiter,5,2026-09-27T07:02:30.633Z,9999-12-31,1
7,280,Pamela Ansman-Wolfe,1,2026-09-27T07:02:30.633Z,9999-12-31,1
8,281,Shu Ito,4,2026-09-27T07:02:30.633Z,9999-12-31,1
9,282,José Saraiva,6,2026-09-27T07:02:30.633Z,9999-12-31,1
10,283,David Campbell,1,2026-09-27T07:02:30.633Z,9999-12-31,1


Total rows: 17


In [0]:
# 写入 Curated Zone
curated_path = "abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/DimSalesPerson/"

dim_salesperson.write \
    .format("delta") \
    .mode("overwrite") \
    .save(curated_path)

print(f"✅ DimSalesPerson written to {curated_path}")

✅ DimSalesPerson written to abfss://curated@stadventureworksdw.dfs.core.windows.net/AdventureWorksDW/DimSalesPerson/


In [0]:
# 验证
df_check = spark.read.format("delta").load(curated_path)
print(f"Total rows: {df_check.count()}")
df_check.show()

Total rows: 17
+--------------+-------------+--------------------+-----------+--------------------+----------+----------+
|SalesPersonKey|SalesPersonID|     SalesPersonName|TerritoryID|          valid_from|  valid_to|is_current|
+--------------+-------------+--------------------+-----------+--------------------+----------+----------+
|             1|          274|       Stephen Jiang|       NULL|2026-09-27 07:02:...|9999-12-31|         1|
|             2|          275|      Michael Blythe|          2|2026-09-27 07:02:...|9999-12-31|         1|
|             3|          276|      Linda Mitchell|          4|2026-09-27 07:02:...|9999-12-31|         1|
|             4|          277|      Jillian Carson|          3|2026-09-27 07:02:...|9999-12-31|         1|
|             5|          278|      Garrett Vargas|          6|2026-09-27 07:02:...|9999-12-31|         1|
|             6|          279|         Tsvi Reiter|          5|2026-09-27 07:02:...|9999-12-31|         1|
|             7|      